In [2]:
import xarray as xr
import icechunk as ic
import os

In [3]:
# Set up your scratch folder path
username       = os.environ["USER"]
glade_scratch  = "/glade/derecho/scratch/" + username
print(glade_scratch)

/glade/derecho/scratch/harshah


In [4]:
conus2d = xr.open_dataset('https://data.gdex.ucar.edu/d559000/kerchunk/wrf2d-remote-https.parq',engine='kerchunk')
conus2d

<xarray.Dataset> Size: 454TB
Dimensions:                 (Time: 376944, south_north: 1015, west_east: 1367,
                             soil_layers_stag: 4, snow_layers_stag: 3,
                             west_east_stag: 1368, south_north_stag: 1016,
                             snso_layers_stag: 7)
Coordinates:
  * Time                    (Time) datetime64[ns] 3MB 1979-10-01 ... 2022-09-...
    XTIME                   (Time) datetime64[ns] 3MB ...
    XLAT                    (Time, south_north, west_east) float32 2TB ...
    XLAT_V                  (Time, south_north_stag, west_east) float32 2TB ...
    XLONG                   (Time, south_north, west_east) float32 2TB ...
    XLONG_V                 (Time, south_north_stag, west_east) float32 2TB ...
    XLAT_U                  (Time, south_north, west_east_stag) float32 2TB ...
    XLONG_U                 (Time, south_north, west_east_stag) float32 2TB ...
Dimensions without coordinates: south_north, west_east, soil_layers_stag,
                                snow_layers_stag, west_east_stag,
                                south_north_stag, snso_layers_stag
Data variables: (12/194)
    ACDEWC                  (Time, south_north, west_east) float32 2TB ...
    ACDRIPR                 (Time, south_north, west_east) float32 2TB ...
    ACDRIPS                 (Time, south_north, west_east) float32 2TB ...
    ACECAN                  (Time, south_north, west_east) float32 2TB ...
    ACEDIR                  (Time, south_north, west_east) float32 2TB ...
    ACETLSM                 (Time, south_north, west_east) float32 2TB ...
    ...                      ...
    index_snow_layers_stag  (Time, snow_layers_stag) int32 5MB ...
    index_snso_layers_stag  (Time, snso_layers_stag) int32 11MB ...
    index_soil_layers_stag  (Time, soil_layers_stag) int32 6MB ...
    totalIce                (Time, south_north, west_east) float32 2TB ...
    totalLiq                (Time, south_north, west_east) float32 2TB ...
    totalVap                (Time, south_north, west_east) float32 2TB ...
Attributes: (12/151)
    TITLE:                            OUTPUT FROM WRF V3.9.1.1 MODEL
    START_DATE:                      1979-10-01_00:00:00
    SIMULATION_START_DATE:           1979-10-01_00:00:00
    WEST-EAST_GRID_DIMENSION:        1368
    SOUTH-NORTH_GRID_DIMENSION:      1016
    BOTTOM-TOP_GRID_DIMENSION:       51
    ...                              ...
    Source_List:                     ./wrf_constants_list_conus404_v20201005.csv
    history:                         Mon Sep 12 21:16:06 2022: ncks -v XLAT_V...
    Source_Code:                     make_conusii_2d.csh
    Project:                         USGS404
    Conventions:                     CF-1.0
    Usage:                           ncgen -k nc4 -o 3_layers_stag.nc 3_layer...

In [ ]:
import icechunk as ic
from obstore.store import HTTPStore
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkParquetParser

url = "https://data.gdex.ucar.edu/"
vds = open_virtual_dataset(
    "https://data.gdex.ucar.edu/d559000/kerchunk/wrf2d-remote-https.parq",
    registry=ObjectStoreRegistry({url: HTTPStore.from_url(url)}),
    parser=KerchunkParquetParser(),
)
vds

In [ ]:
config = ic.RepositoryConfig.default()
config.set_virtual_chunk_container(ic.VirtualChunkContainer(url, ic.http_store()))
repo = ic.Repository.create(ic.local_filesystem_storage("conus404_prec.icechunk"), config=config)
session = repo.writable_session("main")
vds[["PREC_ACC_NC"]].vz.to_icechunk(session.store)
session.commit("PREC_ACC_NC")